In [2]:
import os
import cv2
import tensorflow as tf  
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils import show_img

2025-11-04 17:46:23.458372: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 17:46:23.653849: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 17:46:25.235630: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [6]:
train_dir = "chars_train_data/"

In [13]:
# Load training data
def load_dir_pngs_and_preprocess(directory, preprocess):
    labels = []
    imgs = []
    for (root,dirs,files) in os.walk(directory, topdown=True):
        for file in tqdm(files, desc=f"Loading pngs from '{directory}'"):
            if not file.endswith('.png'):
                continue
            label = file.split("_")[0]
            img_path = os.path.join(root, file)
            labels.append(label)
            imgs.append(preprocess(cv2.imread(img_path)))
    return labels, imgs


In [14]:
def crop_img(img):
    grey = np.mean(img, axis=2)
    coords = np.argwhere(grey < 255)
    if coords.size == 0:
        return grey 

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)

    cropped = grey[y_min-1:y_max+2, x_min-1:x_max+2]
    
    # convert to blck n white
    return np.where(cropped == 255, 0, 255)

In [15]:
labels, imgs = load_dir_pngs_and_preprocess(train_dir, crop_img)

Loading pngs from 'chars_train_data/': 100%|██████████████████████████████████████████████████████████████████████████████████████| 44207/44207 [00:46<00:00, 959.37it/s]
